In [1]:
from pyspark.sql.session import SparkSession

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port","4141")
        .config("spark.port.maxRetries", "0")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):
   
    global spark 
    
    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)


spark = stop_start("working_with_different_data_type")

No existing 'spark' variable found.
Starting new Spark session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/08 17:21:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Working with Nulls in Data

- Always use nulls to represent missing or empty data in your
DataFrames.
- Spark can optimize working with null values more than it can if you use empty
strings or other values.

- use the .na subpackage on a DataFrame.

---

```
There are two things you can do with null values: you can explicitly drop nulls or you can fill
them with a value (globally or on a per-column basis). Let’s experiment with each of these now.
```

---

In [3]:
! ls ../data

bands.json		    more_cars.json     retail-data
bands_readmode_tester.json  movies.json        retail_db
cars_dates.json		    numbers.csv        sampleTextFile.txt
cars.json		    optimization-data  sample_text.txt
flight-data		    output	       stocks.csv
guitarPlayers.json	    pictures	       taxi_zones.csv
guitars.json		    population.json


In [14]:
retail_df = (
    spark
    .read
    .format("csv")
    .option("inferSchema", "false")
    .option("header","true")
    .load("../data/retail-data")
)

retail_df.show(10,False)

+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|537226   |22811    |SET OF 6 T-LIGHTS CACTI       |6       |2010-12-06 08:34:00|2.95     |15987.0   |United Kingdom|
|537226   |21713    |CITRONELLA CANDLE FLOWERPOT   |8       |2010-12-06 08:34:00|2.1      |15987.0   |United Kingdom|
|537226   |22927    |GREEN GIANT GARDEN THERMOMETER|2       |2010-12-06 08:34:00|5.95     |15987.0   |United Kingdom|
|537226   |20802    |SMALL GLASS SUNDAE DISH CLEAR |6       |2010-12-06 08:34:00|1.65     |15987.0   |United Kingdom|
|537226   |22052    |VINTAGE CARAVAN GIFT WRAP     |25      |2010-12-06 08:34:00|0.42     |15987.0   |United Kingdom|
|537226   |22705    |WRAP GREEN PEARS              |25  

In [6]:
retail_df.describe().show() ## mean there are nulls in description , see the count

[Stage 3:==================================================>        (6 + 1) / 7]

+-------+------------------+------------------+--------------------+-----------------+------------------+------------------+--------------+
|summary|         InvoiceNo|         StockCode|         Description|         Quantity|         UnitPrice|        CustomerID|       Country|
+-------+------------------+------------------+--------------------+-----------------+------------------+------------------+--------------+
|  count|             19632|             19632|               19563|            19632|             19632|             12981|         19632|
|   mean| 537171.9999483897|  28512.7373395967|                NULL|7.549765688671557| 7.718832518337488|15606.855943301749|          NULL|
| stddev|433.12482813623495|18365.746858843257|                NULL|77.43844536458644|203.78479896580612|1769.4463259758816|          NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|            -9360|               0.0|           12347.0|     Australia|
|    max|           

In [50]:
from pyspark.sql.functions import coalesce, col
from pyspark.sql.types import Row,  StructType, StructField, StringType, IntegerType



fav_rdd = [ 
    Row("Sujeet","Gorakhpur", "Uttar Pradesh", 32 ),
    Row("Ram","Pune", "Maharashtra", None),
    Row("Shubhash",None, "Paschim Bangal", 24),
    Row("Karishma","Patna", None, None),
    Row("Karishma",None, None, None)
    ]


dist_fav_rdd = spark.sparkContext.parallelize(fav_rdd)

schema = StructType([
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("age", IntegerType(), True)  
])


df = spark.createDataFrame(dist_fav_rdd, schema)
df.show(10, False)

df.printSchema()

+--------+---------+--------------+----+
|name    |city     |state         |age |
+--------+---------+--------------+----+
|Sujeet  |Gorakhpur|Uttar Pradesh |32  |
|Ram     |Pune     |Maharashtra   |NULL|
|Shubhash|NULL     |Paschim Bangal|24  |
|Karishma|Patna    |NULL          |NULL|
|Karishma|NULL     |NULL          |NULL|
+--------+---------+--------------+----+

root
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)



In [39]:

df.select(coalesce(col("city"),col("state"))).show()



+---------------------+
|coalesce(city, state)|
+---------------------+
|            Gorakhpur|
|                 Pune|
|       Paschim Bangal|
|                Patna|
+---------------------+



In [43]:
## ifnull, nullIf, nvl, and nvl2

df.createOrReplaceTempView("test")
spark.sql("""

  SELECT
ifnull(null, 'return_value'),
nullif('value', 'value'),
nvl(null, 'return_value'),
nvl2('not_null', 'return_value', "else_value")
FROM test LIMIT 1
""").show()


+--------------------------+--------------------+-----------------------+----------------------------------------+
|ifnull(NULL, return_value)|nullif(value, value)|nvl(NULL, return_value)|nvl2(not_null, return_value, else_value)|
+--------------------------+--------------------+-----------------------+----------------------------------------+
|              return_value|                NULL|           return_value|                            return_value|
+--------------------------+--------------------+-----------------------+----------------------------------------+



In [46]:
## The default is to drop any row in which any value is null:

df.na.drop().show()

+------+---------+-------------+---+
|  name|     city|        state|age|
+------+---------+-------------+---+
|Sujeet|Gorakhpur|Uttar Pradesh| 32|
+------+---------+-------------+---+



In [47]:
df.na.drop("any").show()

+------+---------+-------------+---+
|  name|     city|        state|age|
+------+---------+-------------+---+
|Sujeet|Gorakhpur|Uttar Pradesh| 32|
+------+---------+-------------+---+



In [51]:
df.show()

+--------+---------+--------------+----+
|    name|     city|         state| age|
+--------+---------+--------------+----+
|  Sujeet|Gorakhpur| Uttar Pradesh|  32|
|     Ram|     Pune|   Maharashtra|NULL|
|Shubhash|     NULL|Paschim Bangal|  24|
|Karishma|    Patna|          NULL|NULL|
|Karishma|     NULL|          NULL|NULL|
+--------+---------+--------------+----+



In [54]:
df.na.drop("all", subset=["city", "age"]).show()   ## drop only if city and age has null vlue

+--------+---------+--------------+----+
|    name|     city|         state| age|
+--------+---------+--------------+----+
|  Sujeet|Gorakhpur| Uttar Pradesh|  32|
|     Ram|     Pune|   Maharashtra|NULL|
|Shubhash|     NULL|Paschim Bangal|  24|
|Karishma|    Patna|          NULL|NULL|
+--------+---------+--------------+----+



In [58]:
df.na.fill("Uknown").show(10, False)

+--------+---------+--------------+----+
|name    |city     |state         |age |
+--------+---------+--------------+----+
|Sujeet  |Gorakhpur|Uttar Pradesh |32  |
|Ram     |Pune     |Maharashtra   |NULL|
|Shubhash|Uknown   |Paschim Bangal|24  |
|Karishma|Patna    |Uknown        |NULL|
|Karishma|Uknown   |Uknown        |NULL|
+--------+---------+--------------+----+



In [62]:
df.na.fill("Uknown", subset=["city", "age"]).show() ## here age is integer , it did not fill that null value for Karishma last record

+--------+---------+--------------+----+
|    name|     city|         state| age|
+--------+---------+--------------+----+
|  Sujeet|Gorakhpur| Uttar Pradesh|  32|
|     Ram|     Pune|   Maharashtra|NULL|
|Shubhash|   Uknown|Paschim Bangal|  24|
|Karishma|    Patna|          NULL|NULL|
|Karishma|   Uknown|          NULL|NULL|
+--------+---------+--------------+----+



In [63]:
df.na.fill("Uknown", subset=["city", "state"]).show()

+--------+---------+--------------+----+
|    name|     city|         state| age|
+--------+---------+--------------+----+
|  Sujeet|Gorakhpur| Uttar Pradesh|  32|
|     Ram|     Pune|   Maharashtra|NULL|
|Shubhash|   Uknown|Paschim Bangal|  24|
|Karishma|    Patna|        Uknown|NULL|
|Karishma|   Uknown|        Uknown|NULL|
+--------+---------+--------------+----+



In [64]:
# Replace the string "Unknown" with None (null) in specific columns
df_cleaned = df.na.replace("Unknown", None, ["city", "state"])

df_cleaned.show()

# Replace a numeric placeholder -1 with None in the age column
df_cleaned = df.na.replace(-1, None, ["age"])


df_cleaned.show()


+--------+---------+--------------+----+
|    name|     city|         state| age|
+--------+---------+--------------+----+
|  Sujeet|Gorakhpur| Uttar Pradesh|  32|
|     Ram|     Pune|   Maharashtra|NULL|
|Shubhash|     NULL|Paschim Bangal|  24|
|Karishma|    Patna|          NULL|NULL|
|Karishma|     NULL|          NULL|NULL|
+--------+---------+--------------+----+

+--------+---------+--------------+----+
|    name|     city|         state| age|
+--------+---------+--------------+----+
|  Sujeet|Gorakhpur| Uttar Pradesh|  32|
|     Ram|     Pune|   Maharashtra|NULL|
|Shubhash|     NULL|Paschim Bangal|  24|
|Karishma|    Patna|          NULL|NULL|
|Karishma|     NULL|          NULL|NULL|
+--------+---------+--------------+----+



In [68]:
# Map multiple old values to new values
replacement_map = {
    "Pune": "New York",
    "Gorakhpur": "Los Angeles",
    "Unknown": None
}

df_cleaned = df.na.replace(replacement_map, subset=["city"])

df_cleaned.show()


+--------+-----------+--------------+----+
|    name|       city|         state| age|
+--------+-----------+--------------+----+
|  Sujeet|Los Angeles| Uttar Pradesh|  32|
|     Ram|   New York|   Maharashtra|NULL|
|Shubhash|       NULL|Paschim Bangal|  24|
|Karishma|      Patna|          NULL|NULL|
|Karishma|       NULL|          NULL|NULL|
+--------+-----------+--------------+----+



In [69]:
spark.stop()